# Snooker Ball Detection - YOLOv8 Training
Train on the Roboflow snooker dataset (17K+ images, 9 classes).

**Setup:** Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Download the Roboflow snooker dataset
from roboflow import Roboflow

rf = Roboflow(api_key='mye5rNUDd96MIiCNLFJ3')
project = rf.workspace('nxera').project('snooker-pocket-and-ball-detection')
version = project.version(1)
dataset = version.download('yolov8', location='/kaggle/working/dataset')
print('Dataset downloaded!')

In [ ]:
# Fix data.yaml paths
import yaml

yaml_path = '/kaggle/working/dataset/data.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['path'] = '/kaggle/working/dataset'
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print('Classes:', data['names'])
print('NC:', data['nc'])

In [ ]:
# Train YOLOv8s - full dataset, 50 epochs
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    patience=15,
    workers=4,
    save=True,
    plots=True,
    verbose=True,
)
print('Training complete!')

In [ ]:
# Show training results
from IPython.display import Image, display
import glob

for img in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    paths = glob.glob(f'runs/detect/train*/{img}')
    if paths:
        display(Image(filename=paths[-1], width=800))

In [ ]:
# Test on validation images
import glob
best_pt = sorted(glob.glob('runs/detect/train*/weights/best.pt'))[-1]
print(f'Best model: {best_pt}')

model = YOLO(best_pt)
metrics = model.val(data=yaml_path)
print(f'mAP50: {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

In [ ]:
# Download the trained model
import shutil
shutil.copy(best_pt, '/kaggle/working/best_color.pt')
print('Model saved to /kaggle/working/best_color.pt')
print('Download this file and copy to:')
print('  final/src/snookervision/data/model/best_color.pt')